# Notebook 53 — Latent Regime Discovery and Meta-Law in Latent Space

Notebook 52 showed that regime structure exists in coefficient space, but that **metadata alone**
does not reliably choose the correct piecewise predictor.

This notebook shifts the meta-law to latent space:

\[
\text{metadata} \to Z \to \beta \to g(r,c)
\]

where:

- `Z` is a low-dimensional latent representation of regime coefficients
- `\beta` is the coefficient vector of the governing equation
- `g(r,c)` is the governing residual-flow field

## Main goals

1. Learn a latent representation of regime coefficients.
2. Predict latent coordinates from metadata.
3. Reconstruct coefficients from latent coordinates.
4. Compare latent-space prediction against direct coefficient prediction.
5. Test whether latent prediction repairs harder blocks from Notebook 52.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.cluster import KMeans

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed template and per-regime coefficient dataset

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(["system", "task", "forcing_mode", "k", "flow_mode"]):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

In [ ]:
# Metadata matrix

meta_df = coef_df[["regime_id", "system", "task", "forcing_mode", "flow_mode", "k"]].copy()
X_base = pd.get_dummies(meta_df[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
X_base["k"] = coef_df["k"].astype(float).values
X_base["k2"] = coef_df["k"].astype(float).values**2
Y_coef = coef_df[COEF_COLS].copy()

display(X_base.head())
display(Y_coef.head())

## Learn latent representation and latent clusters

In [ ]:
coef_scaler = StandardScaler()
Y_std = coef_scaler.fit_transform(Y_coef.to_numpy(dtype=float))

pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(Y_std)

coef_df["z1"] = Z[:, 0]
coef_df["z2"] = Z[:, 1]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
coef_df["latent_cluster"] = kmeans.fit_predict(Z)

print("Explained variance ratio:", pca.explained_variance_ratio_)
display(coef_df[["regime_id", "system", "task", "forcing_mode", "k", "flow_mode", "z1", "z2", "latent_cluster"]].head())

In [ ]:
plt.figure(figsize=(8, 5))
for cl in sorted(coef_df["latent_cluster"].unique()):
    sub = coef_df[coef_df["latent_cluster"] == cl]
    plt.scatter(sub["z1"], sub["z2"], label=f"cluster {cl}")
    for _, r in sub.iterrows():
        plt.text(r["z1"], r["z2"], r["forcing_mode"], fontsize=7, alpha=0.6)
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("Latent regime geometry")
plt.legend()
plt.tight_layout()
plt.show()

## Metadata → latent Z → coefficients

In [ ]:
def fit_predict_linear(X_train, Y_train, X_test):
    model = LinearRegression()
    model.fit(np.asarray(X_train, float), np.asarray(Y_train, float))
    return model.predict(np.asarray(X_test, float))

def fit_predict_ridge(X_train, Y_train, X_test):
    model = Ridge(alpha=1.0)
    model.fit(np.asarray(X_train, float), np.asarray(Y_train, float))
    return model.predict(np.asarray(X_test, float))

def reconstruct_beta_from_z(z_pred, pca, scaler):
    z_pred = np.atleast_2d(np.asarray(z_pred, dtype=float))
    y_std = pca.inverse_transform(z_pred)
    return scaler.inverse_transform(y_std)

def fit_predict_latent_linear(X_train, Z_train, X_test):
    return fit_predict_linear(X_train, Z_train, X_test)

def fit_predict_latent_ridge(X_train, Z_train, X_test):
    return fit_predict_ridge(X_train, Z_train, X_test)

## Compare direct coefficient prediction vs latent prediction

In [ ]:
# Harder blocks reused from Notebook 52

hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
}

beta_shared = Y_coef.to_numpy(dtype=float).mean(axis=0)

compare_rows = []

for block_name, test_mask in hard_block_masks.items():
    train_mask = ~test_mask

    X_train = X_base.loc[train_mask].reset_index(drop=True)
    X_test = X_base.loc[test_mask].reset_index(drop=True)

    Y_train = Y_coef.loc[train_mask].reset_index(drop=True)
    Y_test = Y_coef.loc[test_mask].reset_index(drop=True)

    Z_train = coef_df.loc[train_mask, ["z1", "z2"]].reset_index(drop=True)
    Z_test = coef_df.loc[test_mask, ["z1", "z2"]].reset_index(drop=True)

    test_meta = coef_df.loc[test_mask].reset_index(drop=True)

    for i in range(len(test_meta)):
        regime_id = test_meta.iloc[i]["regime_id"]
        beta_true = Y_test.iloc[i].to_numpy(dtype=float)
        z_true = Z_test.iloc[i].to_numpy(dtype=float)
        x_te = X_test.iloc[[i]]

        # direct prediction
        direct_candidates = {
            "direct_linear": fit_predict_linear(X_train, Y_train, x_te)[0],
            "direct_ridge": fit_predict_ridge(X_train, Y_train, x_te)[0],
        }
        best_direct_name = min(direct_candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, direct_candidates[k])))
        beta_direct = direct_candidates[best_direct_name]

        # latent prediction
        latent_candidates = {
            "latent_linear": reconstruct_beta_from_z(fit_predict_latent_linear(X_train, Z_train, x_te)[0], pca, coef_scaler)[0],
            "latent_ridge": reconstruct_beta_from_z(fit_predict_latent_ridge(X_train, Z_train, x_te)[0], pca, coef_scaler)[0],
        }
        best_latent_name = min(latent_candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, latent_candidates[k])))
        beta_latent = latent_candidates[best_latent_name]

        sub = regime_subsets[regime_id]

        compare_rows.append({
            "block": block_name,
            "regime_id": regime_id,
            "best_direct_model": best_direct_name,
            "best_latent_model": best_latent_name,

            "coef_rmse_direct": math.sqrt(mean_squared_error(beta_true, beta_direct)),
            "coef_rmse_latent": math.sqrt(mean_squared_error(beta_true, beta_latent)),

            "traj_rmse_direct": trajectory_gap(sub, beta_true, beta_direct),
            "traj_rmse_latent": trajectory_gap(sub, beta_true, beta_latent),

            "z_rmse_linear": math.sqrt(mean_squared_error(z_true, fit_predict_latent_linear(X_train, Z_train, x_te)[0])),
            "z_rmse_ridge": math.sqrt(mean_squared_error(z_true, fit_predict_latent_ridge(X_train, Z_train, x_te)[0])),
        })

latent_compare_df = pd.DataFrame(compare_rows)
display(latent_compare_df.head())

In [ ]:
# Summary by harder block

summary_df = latent_compare_df.groupby("block")[[
    "coef_rmse_direct", "coef_rmse_latent",
    "traj_rmse_direct", "traj_rmse_latent",
    "z_rmse_linear", "z_rmse_ridge"
]].mean().reset_index()

display(summary_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
x = np.arange(len(summary_df))
w = 0.35

axes[0].bar(x - w/2, summary_df["coef_rmse_direct"], width=w, label="direct")
axes[0].bar(x + w/2, summary_df["coef_rmse_latent"], width=w, label="latent")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[0].set_title("Coefficient RMSE")
axes[0].legend()

axes[1].bar(x - w/2, summary_df["traj_rmse_direct"], width=w, label="direct")
axes[1].bar(x + w/2, summary_df["traj_rmse_latent"], width=w, label="latent")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[1].set_title("Trajectory RMSE")
axes[1].legend()

axes[2].bar(x - w/2, summary_df["z_rmse_linear"], width=w, label="z linear")
axes[2].bar(x + w/2, summary_df["z_rmse_ridge"], width=w, label="z ridge")
axes[2].set_xticks(x)
axes[2].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[2].set_title("Latent-space RMSE")
axes[2].legend()

plt.tight_layout()
plt.show()

## Latent cluster structure vs metadata

In [ ]:
cluster_counts = coef_df.groupby(["latent_cluster", "forcing_mode"]).size().reset_index(name="count")
display(cluster_counts)

pivot = cluster_counts.pivot(index="latent_cluster", columns="forcing_mode", values="count").fillna(0)
pivot.plot(kind="bar", figsize=(9, 4))
plt.ylabel("count")
plt.title("Latent clusters by forcing mode")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Representative regime comparison

In [ ]:
# Pick regime where latent improves most over direct on trajectory RMSE

rep_idx = int(np.argmax(latent_compare_df["traj_rmse_direct"] - latent_compare_df["traj_rmse_latent"]))
rep = latent_compare_df.iloc[rep_idx]
regime_id = rep["regime_id"]

test_mask = coef_df["regime_id"] == regime_id
train_mask = ~test_mask

X_train = X_base.loc[train_mask].reset_index(drop=True)
X_test = X_base.loc[test_mask].reset_index(drop=True)
Y_train = Y_coef.loc[train_mask].reset_index(drop=True)
Y_test = Y_coef.loc[test_mask].reset_index(drop=True)
Z_train = coef_df.loc[train_mask, ["z1", "z2"]].reset_index(drop=True)

beta_true = Y_test.iloc[0].to_numpy(dtype=float)

beta_direct = fit_predict_linear(X_train, Y_train, X_test)[0]
z_pred = fit_predict_latent_linear(X_train, Z_train, X_test)[0]
beta_latent = reconstruct_beta_from_z(z_pred, pca, coef_scaler)[0]

sub = regime_subsets[regime_id]
cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
rmin, rmax = sub["residual"].min(), sub["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(sub["residual"], 0.1), np.quantile(sub["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

def integrate_beta(beta, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for j in range(len(cgrid) - 1):
        c = cgrid[j]
        dc = float(cgrid[j+1] - cgrid[j])
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        g = float(np.clip(x @ beta, -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
titles = ["Global direct", "Latent-space", "Shared-law", "Regime-specific"]
betas = [beta_direct, beta_latent, beta_shared, beta_true]

for ax, title, beta in zip(axes, titles, betas):
    for r0 in r0s:
        ax.plot(cgrid, integrate_beta(beta, r0))
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Representative regime: {regime_id}", y=1.03)
plt.tight_layout()
plt.show()

## Final verdicts

In [ ]:
def verdict_from_summary(row):
    if row["traj_rmse_latent"] < row["traj_rmse_direct"]:
        if row["coef_rmse_latent"] < row["coef_rmse_direct"]:
            return "latent better"
        return "latent helps on dynamics"
    return "direct better"

summary_df["verdict"] = summary_df.apply(verdict_from_summary, axis=1)
display(summary_df)

## Working conclusion

Notebook 53 tests whether the regime structure found in Notebook 52 is better modeled by
predicting a low-dimensional latent representation first, then reconstructing coefficients.

A strong result is:
- latent-space prediction reduces harder-block error
- latent geometry aligns with regime structure
- metadata predicts latent Z more reliably than it predicts coefficients directly

If that pattern holds on your real exports, the next notebook should be:

**Notebook 54 — hybrid latent + adaptive template selection**